<a href="https://colab.research.google.com/github/NancyMerciline/NLP_EDC/blob/main/Lab5_DeepLearning_LLMs_NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 5: Deep Learning & LLMs for NLP

**Course:** Natural Language Processing


**Objectives:**
- Understand RNN, LSTM, GRU architectures for sequence modeling
- Use pre-trained Transformers for NER
- Interact with LLMs via API for text generation

---

## Instructions

1. Complete all exercises marked with `# YOUR CODE HERE`
2. **Answer all written questions** in the designated markdown cells
3. Save your completed notebook
4. **Push to your Git repository and send the link to: yoroba93@gmail.com**

---

## Lab Structure

| Part | Model | Task |
|------|-------|------|
| A | RNN | Character-level Language Model |
| B | LSTM | Sentiment Analysis |
| C | GRU | News Classification |
| D | Transformer | Named Entity Recognition |
| E | LLM (Mistral) | Text Generation & QA |

---

## Setup

In [1]:
# Install required libraries (uncomment if needed)
# !pip install torch transformers datasets requests numpy pandas matplotlib

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import warnings
warnings.filterwarnings('ignore')

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

Using device: cpu
PyTorch version: 2.11.0+cpu


---

# PART A: RNN - Character-Level Language Model (10 min)

**Use Case:** Predict the next character for autocomplete.

**Dataset:** Tiny Shakespeare

In [2]:
import requests # Add this import

# The datasets library no longer supports loading 'tiny_shakespeare' directly as a script.
# We will download the text file directly.
# Source: https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
response = requests.get(url)
response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)
text = response.text[:10000] # Use first 10K chars for speed

print(f"Text length: {len(text)} characters")
print(f"Sample: {text[:200]}")

Text length: 10000 characters
Sample: First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you


In [3]:
# Create character vocabulary
chars = sorted(list(set(text)))
vocab_size = len(chars)
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for i, ch in enumerate(chars)}

print(f"Vocabulary size: {vocab_size}")
print(f"Characters: {''.join(chars[:30])}...")

Vocabulary size: 57
Characters: 
 !',-.:;?ABCDEFHIJLMNOPRSTUVW...


In [5]:
import torch

# Prepare sequences
seq_length = 30
X, y = [], []

for i in range(len(text) - seq_length):
    X.append([char_to_idx[c] for c in text[i:i+seq_length]])
    y.append(char_to_idx[text[i+seq_length]])

X = torch.tensor(X, dtype=torch.long)
y = torch.tensor(y, dtype=torch.long)

print(f"Sequences: {X.shape[0]}, Sequence length: {seq_length}")

Sequences: 9970, Sequence length: 30


In [8]:
import torch.nn as nn
import torch # Added for torch.device

# Check device (added for self-containment)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Simple RNN model
class CharRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.rnn(x)
        out = self.fc(out[:, -1, :])  # Last timestep
        return out

# Create model
rnn_model = CharRNN(vocab_size, embed_dim=32, hidden_dim=64).to(device)
print(f"RNN Parameters: {sum(p.numel() for p in rnn_model.parameters()):,}")

RNN Parameters: 11,801


### Exercise A.1: Train the RNN

In [11]:
from torch.utils.data import TensorDataset, DataLoader # Added for self-containment
import torch.optim as optim # Added for self-containment

# TODO: Complete the training loop

# Hyperparameters
batch_size = 128
epochs = 5
learning_rate = 0.005  # YOUR CHOICE: 0.001-0.01

# DataLoader
dataset = TensorDataset(X[:5000], y[:5000])  # Use subset for speed
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(rnn_model.parameters(), lr=learning_rate)

# Training loop
losses = []
for epoch in range(epochs):
    total_loss = 0
    for batch_X, batch_y in loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device);

        # YOUR CODE HERE
        # 1. Zero gradients
        optimizer.zero_grad()
        # 2. Forward pass
        outputs = rnn_model(batch_X)
        # 3. Compute loss
        loss = criterion(outputs, batch_y)
        # 4. Backward pass
        loss.backward()
        # 5. Update weights
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(loader)
    losses.append(avg_loss)
    print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")

Epoch 1/5, Loss: 3.0374
Epoch 2/5, Loss: 2.4789
Epoch 3/5, Loss: 2.2597
Epoch 4/5, Loss: 2.1656
Epoch 5/5, Loss: 2.0677


In [12]:
# Generate text
def generate_text(model, start_str, length=100):
    model.eval()
    chars_generated = list(start_str)
    input_seq = [char_to_idx.get(c, 0) for c in start_str[-seq_length:]]

    with torch.no_grad():
        for _ in range(length):
            x = torch.tensor([input_seq[-seq_length:]]).to(device)
            output = model(x)
            pred_idx = torch.argmax(output, dim=1).item()
            chars_generated.append(idx_to_char[pred_idx])
            input_seq.append(pred_idx)

    return ''.join(chars_generated)

# Test generation
print("Generated text:")
print(generate_text(rnn_model, "To be or not", length=100))

Generated text:
To be or not the sor the sor the sor the sor the sor the sor the sor the sor the sor the sor the sor the sor the


---

# PART B: LSTM - Sentiment Analysis

**Use Case:** Classify movie review sentiment.

**Dataset:** IMDB Reviews

In [13]:
# Update datasets and huggingface_hub to resolve HfUriError
!pip install --upgrade datasets huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.1/721.1 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.4 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.20.1
    Uninstalling huggingface_hub-1.20.1:
      Successfully uninstalled huggingface_hub-1.20.1
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [1]:
!pip install -q datasets==3.6.0 huggingface_hub==0.33.5 fsspec==2025.3.0 gcsfs==2025.3.0

In [2]:
from datasets import load_dataset

imdb = load_dataset("imdb")
print(imdb)

# Small sample for quick training
train_texts = imdb['train']['text'][:1000]
train_labels = imdb['train']['label'][:1000]
test_texts = imdb['test']['text'][:200]
test_labels = imdb['test']['label'][:200]

print(f"Train: {len(train_texts)}, Test: {len(test_texts)}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})
Train: 1000, Test: 200


In [3]:
# Simple tokenization and vocabulary
from collections import Counter
import re

def tokenize(text):
    return re.findall(r'\b\w+\b', text.lower())[:100]  # Max 100 tokens

# Build vocabulary from training data
all_tokens = [tok for text in train_texts for tok in tokenize(text)]
vocab = {word: idx+2 for idx, (word, _) in enumerate(Counter(all_tokens).most_common(5000))}
vocab['<PAD>'] = 0
vocab['<UNK>'] = 1

print(f"Vocabulary size: {len(vocab)}")

Vocabulary size: 5002


In [5]:
import torch # Added for self-containment

# Encode texts
def encode_text(text, max_len=100):
    tokens = tokenize(text)
    encoded = [vocab.get(t, 1) for t in tokens]  # 1 = UNK
    padded = encoded[:max_len] + [0] * (max_len - len(encoded))
    return padded[:max_len]

X_train = torch.tensor([encode_text(t) for t in train_texts])
y_train = torch.tensor(train_labels)
X_test = torch.tensor([encode_text(t) for t in test_texts])
y_test = torch.tensor(test_labels)

print(f"Train shape: {X_train.shape}")

Train shape: torch.Size([1000, 100])


### Exercise B.1: Complete the LSTM Model

In [7]:
import torch.nn as nn
import torch # Ensure torch is imported

# Check device (added for self-containment)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# TODO: Complete the LSTM classifier

class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        # YOUR CODE HERE: Define LSTM layer
        # Hint: nn.LSTM(input_size, hidden_size, batch_first=True)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)

        self.fc = nn.Linear(hidden_dim, num_classes)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        x = self.embedding(x)
        x = self.dropout(x)

        # YOUR CODE HERE: Pass through LSTM and get final hidden state
        # Hint: lstm_out, (hidden, cell) = self.lstm(x)
        lstm_out, (hidden, cell) = self.lstm(x)

        out = self.fc(hidden.squeeze(0))  # Use last hidden state
        return out

# Create model
lstm_model = LSTMClassifier(
    vocab_size=len(vocab),
    embed_dim=64,
    hidden_dim=64,
    num_classes=2
).to(device)

print(f"LSTM Parameters: {sum(p.numel() for p in lstm_model.parameters()):,}")

LSTM Parameters: 353,538


In [10]:
from torch.utils.data import TensorDataset, DataLoader # Import TensorDataset and DataLoader
import torch.optim as optim # Added for self-containment

# Quick training
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(lstm_model.parameters(), lr=0.001)

# Train for 3 epochs
for epoch in range(3):
    lstm_model.train()
    total_loss = 0
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        optimizer.zero_grad()
        output = lstm_model(batch_X)
        loss = criterion(output, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")

# Evaluate
lstm_model.eval()
with torch.no_grad():
    test_output = lstm_model(X_test.to(device))
    preds = torch.argmax(test_output, dim=1).cpu()
    acc = (preds == y_test).float().mean()
    print(f"\nTest Accuracy: {acc:.4f}")

Epoch 1, Loss: 0.5217
Epoch 2, Loss: 0.0064
Epoch 3, Loss: 0.0006

Test Accuracy: 1.0000


---

# PART C: GRU - News Classification

**Use Case:** Classify news articles by topic.

**Why GRU?** Fewer parameters than LSTM, faster training.

In [11]:
# Load AG News
ag_news = load_dataset("ag_news")
ag_train = ag_news['train'].shuffle(seed=42).select(range(2000))
ag_test = ag_news['test'].shuffle(seed=42).select(range(500))

ag_labels = {0: 'World', 1: 'Sports', 2: 'Business', 3: 'Sci/Tech'}
print(f"Classes: {list(ag_labels.values())}")

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

Classes: ['World', 'Sports', 'Business', 'Sci/Tech']


### Exercise C.1: Build GRU Classifier

In [13]:
# TODO: Create a GRU classifier (similar to LSTM but using nn.GRU)

class GRUClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        # YOUR CODE HERE: Define GRU layer
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)

        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = self.embedding(x)

        # YOUR CODE HERE: GRU forward pass
        # Note: GRU returns (output, hidden) - no cell state unlike LSTM
        _, hidden = self.gru(x)

        out = self.fc(hidden.squeeze(0))  # Use last hidden state
        return out

# Build vocabulary and encode (reuse tokenize function)
ag_tokens = [tok for item in ag_train for tok in tokenize(item['text'])]
ag_vocab = {word: idx+2 for idx, (word, _) in enumerate(Counter(ag_tokens).most_common(5000))}
ag_vocab['<PAD>'] = 0
ag_vocab['<UNK>'] = 1

def encode_ag(text, vocab, max_len=50):
    tokens = tokenize(text)
    encoded = [vocab.get(t, 1) for t in tokens]
    return (encoded[:max_len] + [0] * max_len)[:max_len]

X_ag_train = torch.tensor([encode_ag(item['text'], ag_vocab) for item in ag_train])
y_ag_train = torch.tensor([item['label'] for item in ag_train])
X_ag_test = torch.tensor([encode_ag(item['text'], ag_vocab) for item in ag_test])
y_ag_test = torch.tensor([item['label'] for item in ag_test])

print(f"AG News - Train: {X_ag_train.shape}, Test: {X_ag_test.shape}")

AG News - Train: torch.Size([2000, 50]), Test: torch.Size([500, 50])


In [14]:
# Create and train GRU model
gru_model = GRUClassifier(
    vocab_size=len(ag_vocab),
    embed_dim=64,
    hidden_dim=64,
    num_classes=4
).to(device)

print(f"GRU Parameters: {sum(p.numel() for p in gru_model.parameters()):,}")
print(f"(Compare to LSTM: GRU has fewer parameters!)")

GRU Parameters: 345,348
(Compare to LSTM: GRU has fewer parameters!)


---

# PART D: Transformer - Named Entity Recognition

**Use Case:** Extract entities from text.

**Dataset:** CoNLL-2003

In [1]:
# Use pre-trained NER model from Hugging Face
!pip install --upgrade transformers huggingface_hub # Ensure compatible versions
from transformers import pipeline

# Load NER pipeline (uses BERT-based model)
print("Loading NER model...")
ner_pipeline = pipeline("ner", model="dslim/bert-base-NER", aggregation_strategy="simple")
print("Model loaded!")

Loading NER model...


config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Model loaded!


In [2]:
# Example NER
text = "Apple Inc. was founded by Steve Jobs in Cupertino, California. Tim Cook is the current CEO."

entities = ner_pipeline(text)
print(f"Text: {text}\n")
print("Entities found:")
for ent in entities:
    print(f"  {ent['word']:20} -> {ent['entity_group']:10} (score: {ent['score']:.3f})")

Text: Apple Inc. was founded by Steve Jobs in Cupertino, California. Tim Cook is the current CEO.

Entities found:
  Apple Inc            -> ORG        (score: 0.999)
  Steve Jobs           -> PER        (score: 0.903)
  Cupertino            -> LOC        (score: 0.998)
  California           -> LOC        (score: 0.999)
  Tim Cook             -> PER        (score: 1.000)


### Exercise D.1: NER on Your Own Texts

In [3]:
import torch # Ensure torch is imported

# TODO: Write 3 sentences and extract entities
# Include: people, organizations, locations

my_sentences = [
    "Elon Musk, CEO of SpaceX, announced a new mission to Mars from Cape Canaveral.",
    "Dr. Sarah Connor, a scientist at Google, presented her research in London.",
    "The United Nations held a climate change summit in New York City with delegates from around the world."
]

for sent in my_sentences:
    if sent != "___":
        print(f"\nText: {sent}")
        entities = ner_pipeline(sent)
        for ent in entities:
            print(f"  {ent['word']:20} -> {ent['entity_group']}")


Text: Elon Musk, CEO of SpaceX, announced a new mission to Mars from Cape Canaveral.
  El                   -> PER
  ##on Musk            -> PER
  SpaceX               -> ORG
  Mars                 -> LOC
  Cape Canaveral       -> LOC

Text: Dr. Sarah Connor, a scientist at Google, presented her research in London.
  Sarah Connor         -> PER
  Google               -> ORG
  London               -> LOC

Text: The United Nations held a climate change summit in New York City with delegates from around the world.
  United Nations       -> ORG
  New York City        -> LOC


---

# PART E: LLM - Text Generation with Mistral API

**Use Case:** Conversational AI and Question Answering.

**Setup:** Get a free API key from https://console.mistral.ai/

In [4]:
# TODO: Enter your Mistral API key
# Get free key at: https://console.mistral.ai/

MISTRAL_API_KEY = "EJeXmWHWgDNt3DYOTFUiXGKj1A8Aqbff"  # YOUR API KEY HERE

In [5]:
import requests

def query_mistral(prompt, max_tokens=150):
    """Query Mistral API."""
    url = "https://api.mistral.ai/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {MISTRAL_API_KEY}",
        "Content-Type": "application/json"
    }
    data = {
        "model": "mistral-small-latest",
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": max_tokens
    }

    response = requests.post(url, headers=headers, json=data)
    if response.status_code == 200:
        return response.json()['choices'][0]['message']['content']
    else:
        return f"Error: {response.status_code} - {response.text}"

# Test (only if API key is set)
if MISTRAL_API_KEY != "___":
    response = query_mistral("What is NLP in one sentence?")
    print(f"Mistral: {response}")
else:
    print("Please set your MISTRAL_API_KEY above.")

Mistral: Natural Language Processing (NLP) is a field of AI that enables computers to understand, interpret, and generate human language.


### Exercise E.1: Compare LLM with Traditional Models

In [6]:
# TODO: Ask Mistral to perform sentiment analysis and compare with our LSTM

test_review = "This movie was absolutely terrible. The acting was bad and the plot made no sense."

# LLM approach
if MISTRAL_API_KEY != "___":
    prompt = f"""Classify the sentiment of this review as 'positive' or 'negative'.
Just respond with one word.

Review: {test_review}

Sentiment:"""

    llm_result = query_mistral(prompt, max_tokens=10)
    print(f"LLM Sentiment: {llm_result}")

# Traditional LSTM approach (if model trained)
try:
    encoded = torch.tensor([encode_text(test_review)]).to(device)
    lstm_model.eval()
    with torch.no_grad():
        lstm_pred = torch.argmax(lstm_model(encoded)).item()
    print(f"LSTM Sentiment: {'positive' if lstm_pred == 1 else 'negative'}")
except:
    print("LSTM model not available")

LLM Sentiment: negative
LSTM model not available


In [7]:
# TODO: Use LLM for summarization (something traditional models can't easily do)

long_text = """
Natural language processing (NLP) is a subfield of linguistics, computer science,
and artificial intelligence concerned with the interactions between computers and
human language, in particular how to program computers to process and analyze large
amounts of natural language data. The result is a computer capable of understanding
the contents of documents, including the contextual nuances of the language within them.
"""

if MISTRAL_API_KEY != "___":
    summary_prompt = f"Summarize this in one sentence:\n\n{long_text}"
    summary = query_mistral(summary_prompt, max_tokens=50)
    print(f"Summary: {summary}")

Summary: Natural language processing (NLP) is a field at the intersection of linguistics, computer science, and AI that enables computers to process, analyze, and understand human language, including its contextual nuances.


---

## Final Written Questions (Personal Interpretation)

Answer these questions based on YOUR experiments:

### Question 1: Model Architecture Comparison

Compare the parameter counts you observed:
- RNN: 11,801 parameters
- LSTM: 353,538 parameters  
- GRU: 345,348 parameters

**Why does LSTM have more parameters than GRU?** (Hint: think about gates)

**YOUR ANSWER:**

LSTM (Long Short-Term Memory) networks generally have more parameters than GRU (Gated Recurrent Unit) networks because LSTMs utilize more gates.

An LSTM unit typically consists of three gates: an input gate, a forget gate, and an output gate. Each of these gates controls the flow of information into and out of the cell state, and each gate involves its own set of weight matrices and bias vectors.

A GRU unit, on the other hand, has only two gates: a reset gate and an update gate. These two gates combine the functionality of the LSTM's input and forget gates into a single update gate, and the output gate is also merged into the update gate and reset gate. This reduction in the number of gates directly translates to fewer weight matrices and bias vectors, resulting in a lower parameter count compared to LSTMs.

### Question 2: RNN vs LSTM for Long Sequences

**Why would LSTM perform better than vanilla RNN for sentiment analysis on long reviews?** Explain the vanishing gradient problem.

**YOUR ANSWER:**

LSTM (Long Short-Term Memory) networks generally perform much better than vanilla RNNs (Recurrent Neural Networks) for sentiment analysis on long reviews primarily due to their ability to mitigate the **vanishing gradient problem** and capture long-term dependencies in sequences.

### Vanishing Gradient Problem in Vanilla RNNs

In vanilla RNNs, information is passed sequentially from one time step to the next. During backpropagation, the gradients are calculated and propagated backward through time. However, as the sequence length increases, the gradients tend to shrink exponentially. This phenomenon is known as the **vanishing gradient problem**.

*   **Effect:** When gradients become very small, the updates to the weights of the initial layers in the network become negligible. This means that the RNN essentially forgets information from earlier parts of the sequence, making it difficult to learn long-term dependencies. For a long movie review, the sentiment might be determined by a crucial word or phrase that appeared much earlier in the text, but a vanilla RNN would struggle to retain that information.

### How LSTMs Address the Problem

LSTMs are specifically designed to overcome the vanishing gradient problem through their sophisticated internal structure, primarily involving a **cell state** and several **gates**:

1.  **Cell State:** The cell state acts as a

### Question 3: Traditional Models vs LLMs

Based on your experiments:
1. **What can LLMs do that LSTM/GRU cannot?**
2. **What are the disadvantages of using LLM APIs?** (Think: cost, latency, privacy)
3. **When would you choose a traditional model over an LLM?**

**YOUR ANSWER:**

1.  **What can LLMs do that LSTM/GRU cannot?**
    LLMs (Large Language Models) can perform a vast array of complex natural language understanding and generation tasks that traditional models like LSTM/GRU cannot, or can only do with extensive task-specific training and engineering. Key capabilities include:
    *   **Zero-shot/Few-shot Learning:** LLMs can generalize to new tasks without explicit fine-tuning or with only a few examples, leveraging their vast pre-training knowledge. Traditional models require substantial labeled datasets for each new task.
    *   **Text Generation:** LLMs can generate coherent, contextually relevant, and creative text, such as poems, stories, articles, and summaries (as demonstrated in Exercise E.1). LSTM/GRU are primarily discriminative models and excel at sequence prediction or classification, not open-ended generation.
    *   **Complex Reasoning and Question Answering:** LLMs can answer complex questions that require understanding context, performing multi-step reasoning, and synthesizing information, often across various domains. Traditional models are typically limited to specific QA formats they were trained on.
    *   **Summarization (Abstractive):** LLMs can produce abstractive summaries, rephrasing and generating new sentences that capture the essence of a longer text. Traditional models like sequence-to-sequence LSTMs usually perform extractive summarization, picking key sentences directly from the source.
    *   **Multilingual Capabilities:** Many LLMs are trained on vast multilingual datasets and can perform tasks across multiple languages without specific models for each language.
    *   **Conversational AI:** LLMs can maintain long-form conversations, track dialogue state, and generate human-like responses, far beyond the capabilities of most traditional sequence models.

2.  **What are the disadvantages of using LLM APIs?** (Think: cost, latency, privacy)
    While powerful, using LLM APIs comes with several disadvantages:
    *   **Cost:** API calls typically incur a per-token or per-request cost, which can quickly become expensive for high-volume applications or long texts. This makes them less economical for tasks that can be handled by simpler models.
    *   **Latency:** Sending requests to an external API, waiting for processing, and receiving a response introduces network latency. For real-time applications or those requiring very fast inference, this can be a significant bottleneck compared to running a local, optimized traditional model.
    *   **Privacy and Data Security:** Sending sensitive or proprietary data to a third-party API provider raises privacy and security concerns. Organizations with strict data governance policies may be hesitant to transmit certain types of information externally.
    *   **Dependence on External Service:** Reliability and availability are subject to the API provider's service. Downtime, rate limits, or changes in API policies can disrupt applications.
    *   **Lack of Customization/Control:** While some APIs offer fine-tuning, the level of control over the model's architecture, training process, or internal workings is limited compared to training and deploying your own model.
    *   **Reproducibility:** API models can be updated by the provider, leading to changes in behavior or performance over time, making it hard to ensure strict reproducibility.

3.  **When would you choose a traditional model over an LLM?**
    Despite the advancements in LLMs, traditional models (like LSTM, GRU, or even simpler ML models) are often preferred in specific scenarios:
    *   **Resource Constraints:** For applications deployed on edge devices, mobile phones, or systems with limited computational power and memory, traditional models with fewer parameters and lower computational requirements are more suitable.
    *   **Cost Sensitivity:** When budget is a primary concern, or for high-throughput tasks where each API call adds up, a locally deployed traditional model can be significantly more cost-effective.
    *   **Low Latency Requirements:** For tasks needing near-instantaneous responses (e.g., real-time fraud detection, immediate sentiment feedback), the overhead of an API call makes LLMs impractical. Traditional models can offer much lower inference times.
    *   **Data Privacy and Security:** In regulated industries (e.g., healthcare, finance) or when dealing with highly sensitive personal data, keeping data within an organization's controlled environment by using local traditional models is often a non-negotiable requirement.
    *   **Domain Specificity with Limited Data:** If a task is very niche and there's a good amount of labeled data for that specific domain, a fine-tuned traditional model might outperform a general-purpose LLM or be more robust.
    *   **Interpretability and Explainability:** Simpler traditional models are often easier to interpret and debug, which can be crucial in applications where understanding *why* a decision was made is important.
    *   **Simpler Tasks:** For straightforward tasks like binary classification (e.g., spam detection, simple sentiment classification) where a traditional model performs adequately, using an LLM would be overkill and inefficient.

---

## Summary

| Model | Strength | Weakness | Best For |
|-------|----------|----------|----------|
| RNN | Simple, fast | Vanishing gradients | Short sequences |
| LSTM | Long-term memory | More parameters | Long text classification |
| GRU | Efficient, fast | Less expressive | When speed matters |
| Transformer | Parallel, contextual | Expensive | NER, QA, many tasks |
| LLM | Versatile, zero-shot | API cost, latency | Complex reasoning |

---

## Submission

- [ ] All code exercises completed
- [ ] All written questions answered
- [ ] Mistral API tested (or explained why not)
- [ ] **Push to Git and send link to: yoroba93@gmail.com**